## Bibliotecas

In [ ]:
# Bibliotecas para manipulación de datos
import pandas as pd
import numpy as np
# Bibliotecas para expresiones regulares
import re 
# Silenciar advertencias
import warnings
warnings.filterwarnings("ignore")
# Bibliotecas para estadística y análisis de datos
from sklearn.cluster import KMeans
# Bibliotecas para conexión con Supabase
import ssl
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy.pool import NullPool

## DATASET (Conexión a Supabase)

In [6]:
# Parámetros de conexión a Supabase, leídos desde .env (nunca escribir credenciales aquí)
import os
from dotenv import load_dotenv
load_dotenv()
DB_USER = os.environ["DB_USER"]
DB_PASS = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = int(os.getenv("DB_PORT", "6543"))
DB_NAME = os.getenv("DB_NAME", "postgres")

# Construcción limpia de la URL de conexión a la base de datos
db_url = URL.create(
    drivername="postgresql+pg8000",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

# Configuración SSL para el pooler
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

engine = create_engine(
    db_url,
    connect_args={"ssl_context": ssl_ctx},
    poolclass=NullPool
)

print("Conectando a Supabase...")

# Dataset de entrenamiento
query_train = """
    SELECT *
    FROM public.matriz_caracteristicas
    WHERE conjunto = 'ENTRENAMIENTO'
    ORDER BY id_poligono;
"""
df_train = pd.read_sql(query_train, engine)
print(f"Set de Entrenamiento cargado: {df_train.shape} (filas, columnas)")

# Dataset de predicción
query_pred = """
    SELECT *
    FROM public.matriz_caracteristicas
    WHERE conjunto = 'PREDICCION'
    ORDER BY id_poligono;
"""
df_pred = pd.read_sql(query_pred, engine)
print(f"Set de Predicción cargado: {df_pred.shape} (filas, columnas)")

# DataFrame concatenado de entrenamiento y predicción
# Nota: el conjunto de predicción trae el objetivo vacío; se silencia el aviso de pandas al concatenar
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    df = pd.concat([df_train, df_pred], ignore_index=True)

# Ordenar por id_poligono
df = df.sort_values("id_poligono", ignore_index=True)
print(f"Dataset completo: {df.shape} (filas, columnas)")


Conectando a Supabase...
Set de Entrenamiento cargado: (138, 98) (filas, columnas)
Set de Predicción cargado: (59, 98) (filas, columnas)
Dataset completo: (197, 98) (filas, columnas)


# **Preprocesamiento de Datos**
## 1. Dataset_Greddy
### Selección de columnas (Línea base con ensayo de Random Forest)
Derivado del análisis exploratorio, determinamos las siguientes acciones para un modelo en Random Forest con 50 variables predictoras.

1. Como las 59 parcelas de predicción están en los mismos municipios que el entrenamiento, la CV aleatoria es válida para este split. 
2. Se crea la una nueva variable `grupo_rend` como una variable estratificadora de 5 splits para separar el efecto de zona.
4. Se incluye `diag_y_resolucion` para tratar el ruido de etiqueta.
5. Se descartan las variables diagnóstico`diag_fenologia_discrepante`, `diag_topo_nodata`, `diag_sin_ciclo`, `diag_ciclo_ambiguo`, `diag_s2_sin_ciclo` y `diag_era5_vecino` como variables de control para Random Forest.
6. Se incluye `diag_topo_pocos_pix` como *feature*.
7. No se consideran las variables predictoras con redundancias exactas (`pl_eos20_doy`, `ac_dias_lluvia_ciclo`, `pl_sos20_doy`, `era5_rs_ciclo_mj_m2`, `pend_p90_grados`, `s2_ndvi_int_abr_oct`, `pl_ndvi_pico`, `s2_centro_pico_doy`, `ac_lluvia_llen_mm`, `era5_apar_ciclo_mj_m2`, `ac_lluvia_ciclo_anom_mm`)

Columnas definidas:
- **Metadatos:** `id_poligono`, `municipio`, `estado`, `conjunto`, `clima_pixel_id`
- **Variable objetivo:** `rendimiento_t_ha` (sin transformación: Random Forest no requiere normalidad)
- **Variables predictoras:** 50 variables (se excluyen las redundancias exactas y las variables podadas)
- **Variable diagnóstico:** `diag_topo_pocos_pix` (binaria 1/0, se usa como *feature*)
- **Variable estratificadora:** `grupo_rend` (KMeans con 2 clusters sobre el rendimiento del set de entrenamiento)
- **Variable ponderadora:** `diag_y_resolucion` (para `sample_weight` en `.fit()`)


In [7]:
# Variable objetivo
TARGET = "rendimiento_t_ha"

# Variables de metadatos (5 identificadores)
METADATOS = ["id_poligono", "municipio", "estado", "conjunto", "clima_pixel_id"]

# Variables predictoras (50 features)
PREDICTORAS = [
    "era5_vpd_medio_flor_kpa", "ac_lluvia_pre_mm", "era5_sm2_flor", "soil_agua_disp_mm", "soil_soc_g_kg",
    "pl_eos50_doy", "soil_densidad_g_cm3", "ac_gdd_ciclo", "pl_amplitud", "soil_limo_pct",
    "era5_vpd_medio_llen_kpa", "pl_ndvi_base", "pl_ndvi_int_abr_oct", "ac_gdd_hasta_flor", "era5_sm2_min_llen",
    "elev_media_m", "pl_dias_meseta", "era5_tmax_media_flor_c", "area_ha_geod", "pl_lai_int_ciclo",
    "pend_max_grados", "pend_media_grados", "pl_lai_max_ciclo", "pl_centro_pico_doy", "pl_tasa_subida",
    "soil_arena_pct", "s2_sos50_doy", "pl_caida_intra_ciclo", "pl_sos50_doy", "era5_sm2_llen",
    "ac_lluvia_veg_mm", "era5_sm1_siembra", "ac_racha_seca_max_flor", "era5_apar_post_pico_mj_m2", "s2_ndvi_pico",
    "ac_lluvia_ciclo_anom_rel", "ac_balance_llen_mm", "elev_std_m", "ac_balance_flor_mm", "dz_pixel_m",
    "pl_n_picos", "soil_ph", "ac_balance_ciclo_mm", "ac_lluvia_max_dia_ciclo_mm", "ac_racha_seca_max_ciclo",
    "pl_integral_ciclo", "ac_lluvia_ciclo_mm", "soil_arcilla_pct", "s2_integral_ciclo", "pl_tasa_bajada",
]

# Variable diagnóstico (se usa como feature binaria)
DIAGNOSTICO = ["diag_topo_pocos_pix"]

# Variable estratificadora (se crea en la siguiente celda)
ESTRATIFICADORA = ["grupo_rend"]

# Variable ponderadora (sample_weight)
PONDERADORA = ["diag_y_resolucion"]

# Redundancias exactas que no entran como predictoras
REDUNDANTES = [
    "pl_eos20_doy", "ac_dias_lluvia_ciclo", "pl_sos20_doy", "era5_rs_ciclo_mj_m2", "pend_p90_grados",
    "s2_ndvi_int_abr_oct", "pl_ndvi_pico", "s2_centro_pico_doy", "ac_lluvia_llen_mm",
    "era5_apar_ciclo_mj_m2", "ac_lluvia_ciclo_anom_mm",
]

COLUMNAS_RF = METADATOS + [TARGET] + PREDICTORAS + DIAGNOSTICO + ESTRATIFICADORA + PONDERADORA

# Validaciones de la selección
assert len(PREDICTORAS) == len(set(PREDICTORAS)) == 50, "Predictoras duplicadas o incompletas"
assert not set(PREDICTORAS) & set(REDUNDANTES), "Hay redundancias exactas en las predictoras"
faltantes = [c for c in COLUMNAS_RF if c not in df.columns and c not in ESTRATIFICADORA]
assert not faltantes, f"Columnas no encontradas en el dataset maestro: {faltantes}"
print(f"Columnas seleccionadas: {len(COLUMNAS_RF)} | Predictoras: {len(PREDICTORAS)}")

Columnas seleccionadas: 59 | Predictoras: 50


In [10]:
# Variable estratificadora: clustering del objetivo (KMeans, 2 grupos) sobre el set de entrenamiento
mask_train = df["conjunto"] == "ENTRENAMIENTO"
y_train = df.loc[mask_train, TARGET]
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(y_train.to_frame())

# El set de predicción no tiene rendimiento, por lo que su grupo queda vacío (NaN)
df["grupo_rend"] = np.nan
df.loc[mask_train, "grupo_rend"] = np.where(km.labels_ == np.argmax(km.cluster_centers_.ravel()), "Alto", "Bajo")
print(df.loc[mask_train].groupby("grupo_rend")[TARGET].describe().round(2))
print("\nUmbral aproximado entre grupos:", round(np.mean(sorted(km.cluster_centers_.ravel())), 2), "t/ha")

            count  mean   std   min   25%   50%   75%   max
grupo_rend                                                 
Alto         80.0  4.60  0.35  3.91  4.45  4.54  4.69  5.59
Bajo         58.0  3.07  0.43  2.00  3.00  3.00  3.50  3.81

Umbral aproximado entre grupos: 3.83 t/ha


In [11]:
# Filtrar el dataset maestro con las columnas de la línea base de Random Forest
df_rf = df[COLUMNAS_RF].copy()

# diag_topo_pocos_pix como columna binaria (1/0)
df_rf["diag_topo_pocos_pix"] = df_rf["diag_topo_pocos_pix"].astype(int)
print(f"Dataset para Random Forest: {df_rf.shape} (filas, columnas)")

Dataset para Random Forest: (197, 59) (filas, columnas)


## Verificación rápida

In [ ]:
# Verificación rápida del DataFrame resultante
df_rf.info()
print(f"\nDimensiones: {df_rf.shape} (filas, columnas)")
nulos = df_rf.isna().sum()
print(f"\nConteo de nulos por columna:\n{nulos[nulos > 0]}")
print(f"\nTotal de nulos: {nulos.sum()}")
print(f"\nNulos por conjunto:\n{df_rf.drop(columns='conjunto').isna().groupby(df_rf['conjunto']).sum().sum(axis=1)}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 59 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_poligono                 197 non-null    object 
 1   municipio                   197 non-null    object 
 2   estado                      197 non-null    object 
 3   conjunto                    197 non-null    object 
 4   clima_pixel_id              197 non-null    object 
 5   rendimiento_t_ha            138 non-null    float64
 6   era5_vpd_medio_flor_kpa     197 non-null    float64
 7   ac_lluvia_pre_mm            197 non-null    float64
 8   era5_sm2_flor               197 non-null    float64
 9   soil_agua_disp_mm           197 non-null    float64
 10  soil_soc_g_kg               197 non-null    float64
 11  pl_eos50_doy                197 non-null    int64  
 12  soil_densidad_g_cm3         197 non-null    float64
 13  ac_gdd_ciclo                197 non

## Exportación

In [ ]:
# Exportar la entrada del Random Forest a data/processed/
from pathlib import Path
SALIDA = Path("..") / "data" / "processed" / "rf_entrada_50vars.csv"
SALIDA.parent.mkdir(parents=True, exist_ok=True)
df_rf.to_csv(SALIDA, index=False)
print(f"Archivo exportado: {SALIDA} {df_rf.shape}")

Archivo exportado: dataset_greedy.csv (197, 59)
